# 02 · Características preprocesadas para semana 2 (CPU)
Genera un único bundle `paper95` para los 88 sujetos: 36 AD, 29 CN y 23 FTD. No descarga ni procesa EEG crudo.

In [ ]:
from pathlib import Path
import os, re, shutil, subprocess, sys, zipfile, kagglehub
assert Path('/kaggle/working').exists() and os.environ.get('KAGGLE_KERNEL_RUN_TYPE')
REPO = Path('/kaggle/working/fpsd-cnn-reassessment')
if not (REPO/'pyproject.toml').is_file():
    def package_version(path):
        match = re.search(r'-v(\d+)(?:\.zip)?$', path.name)
        return int(match.group(1)) if match else 0
    candidates = [p.parent for p in Path('/kaggle/input').glob('**/pyproject.toml') if (p.parent/'src/fpsd_reassessment').is_dir()]
    if candidates:
        selected = max(candidates, key=package_version)
        print('Directorio de código seleccionado:', selected.name)
        shutil.copytree(selected, REPO, dirs_exist_ok=True)
    else:
        code_data = Path(kagglehub.dataset_download('sarahvasquez97/fpsd-cnn-reassessment-code'))
        archives = list(code_data.glob('fpsd-cnn-reassessment-code-v*.zip'))
        if archives:
            REPO.mkdir(parents=True, exist_ok=True)
            selected = max(archives, key=package_version)
            print('Paquete de código seleccionado:', selected.name)
            with zipfile.ZipFile(selected) as archive: archive.extractall(REPO)
        elif (code_data/'pyproject.toml').is_file():
            shutil.copytree(code_data, REPO, dirs_exist_ok=True)
        else:
            raise FileNotFoundError(f'No se encontró un paquete de código válido en {code_data}')
assert (REPO/'scripts/prepare_features.py').is_file()
sys.path.insert(0, str(REPO/'src'))
os.environ['PYTHONPATH'] = str(REPO/'src')
print('Repositorio:', REPO)

In [ ]:
DATASET_DIR = Path('/kaggle/input/datasets/thngdngvn/openneuro-ds004504/ds004504')
assert (DATASET_DIR/'participants.tsv').is_file() and (DATASET_DIR/'derivatives').is_dir()
subprocess.run([sys.executable, str(REPO/'scripts/prepare_features.py'), '--dataset-dir', str(DATASET_DIR), '--output-dir', '/kaggle/working/fpsd_features_all', '--config', str(REPO/'configs/features/paper95.yaml'), '--skip-download', '--diagnoses', 'AD', 'CN', 'FTD'], cwd=str(REPO), check=True)

In [ ]:
import numpy as np, pandas as pd
bundle = Path('/kaggle/working/fpsd_features_all')
with np.load(bundle/'paper95_features.npz') as archive:
    assert archive['X'].shape[1:] == (19, 5) and archive['X_flat'].shape[1] == 95
    assert np.isfinite(archive['X']).all()
metadata = pd.read_parquet(bundle/'windows_metadata.parquet')
qc = pd.read_csv(bundle/'subjects_qc.csv')
counts = metadata.drop_duplicates('subject_id').diagnosis.value_counts().to_dict()
assert counts == {'AD': 36, 'CN': 29, 'FTD': 23}, counts
assert len(qc) == 88 and Path('/kaggle/working/fpsd_features_all_bundle.zip').is_file()
print('Bundle listo:', '/kaggle/working/fpsd_features_all_bundle.zip', counts)